# 🤖 Week 3 Assignment — RAG-Powered Console Chatbot
### WnCC Machine Learning Learner Space 2026

---

This week's assignment has three parts of increasing difficulty.

**Part A — Concept Check:** Short questions testing your understanding of the week's theory.
**Part B — Core Implementation:** Build a complete RAG pipeline from scratch and wire it to an LLM.
**Part C — The Challenge:** Make your chatbot smarter with multi-turn memory and source citation.

**Rules:**
- All `# TODO` blocks must be completed. Read the docstring above each one carefully.
- Run every **Sanity Check** cell after completing a TODO. Fix failures before moving on.
- Part A answers go in the Markdown cells provided — write proper explanations, not one-liners.
- Parts B and C are graded on correctness (sanity checks), code quality, and your reflection.

---
**Run the setup cell first.**

In [18]:
# ========== SETUP (run this first) ==========
!pip install "transformers<5.0.0" torch faiss-cpu sentence-transformers -q

# Authenticate with HuggingFace
# Get your free token at: https://huggingface.co/settings/tokens
from huggingface_hub import login
HF_TOKEN = "hf_smGCVsTyMONWtnxxxxxxxxxxxxxxx"   # <-- paste your token here
login(HF_TOKEN)

import torch
import numpy as np
import faiss
import textwrap
from transformers import pipeline

print('Setup complete!')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

Setup complete!
Device: GPU


---
## Part A — Concept Check

Answer each question in the Markdown cell below it. **2–4 sentences each.** Be precise.

These are not trick questions — they test whether you read the material, not whether you can Google.

---
### A1. Self-Attention

The self-attention formula is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In your own words: what does the matrix $QK^T$ represent before the softmax is applied? What does it become after the softmax? And what does multiplying by $V$ finally produce?

### A1. Self-Attention
**Your answer:**
The matrix $QK^T$ represents the raw alignment or similarity scores between every pair of tokens in a sequence, determining how much 'focus' one word should have on another. Once the softmax is applied, these scores become a probability distribution (weights that sum to 1), indicating the relative importance of each token's context. Finally, multiplying by $V$ produces a weighted sum of the value vectors, creating a new representation for each token that is enriched by information from its most relevant neighbors.

---
### A2. Encoder vs Decoder

Your manager asks you to build two systems:
1. A tool that reads customer support tickets and tags them as `billing`, `technical`, or `general`.
2. A tool that automatically drafts a reply to each ticket.

Which Transformer family (encoder-only, decoder-only, or encoder-decoder) would you use for each, and why? Be specific about what architectural property makes each one suited to its task.

### A2. Encoder vs Decoder
**Your answer:**
1. For tagging customer support tickets, I would use an **encoder-only** family (like BERT) because its bidirectional nature allows it to look at the entire context of a sentence simultaneously to understand intent and category.
2. For drafting replies, I would use a **decoder-only** (like GPT) or **encoder-decoder** (like T5) family. Decoder architectures use causal attention to generate text token-by-token, which is essential for producing coherent, multi-word natural language responses.

---
### A3. RLHF

Explain in your own words why RLHF uses human *comparisons* ("which of these two responses is better?") rather than human-written *ideal responses* ("write the perfect answer to this question"). What practical problem does this solve?

### A3. RLHF
**Your answer:**
RLHF uses comparisons because it is significantly easier and faster for humans to rank which of two AI responses is 'better' or 'safer' than it is for them to write a perfect, high-quality response from scratch for every prompt. This solves the data bottleneck problem and provides a more scalable way to capture subjective human nuances like tone, helpfulness, and safety that are hard to define mathematically.

---
### A4. RAG vs Fine-tuning

A startup has 10,000 internal company documents (policies, FAQs, product specs) and wants to build a Q&A bot over them. The documents are updated weekly.

They are deciding between: (a) fine-tuning an LLM on all the documents, or (b) building a RAG system.

Make the case for RAG. Give at least two concrete reasons why it is better suited to this scenario.

### A4. RAG vs Fine-tuning
**Your answer:**
RAG is superior here because it allows the system to remain up-to-date with weekly policy changes simply by updating the document index, whereas fine-tuning would require expensive and frequent retraining of the model. Additionally, RAG provides transparency through source citations (allowing employees to verify the original document) and significantly reduces the risk of 'hallucination' by forcing the model to stick to the provided context.

---
## Part B — Core Implementation

You will build a **RAG-powered chatbot** in stages:

```
Documents → Chunk → Embed → FAISS Index   (offline, build once)
                                  │
User question → Embed → Retrieve top-k chunks → Build prompt → LLM → Answer
```

The document corpus, embedding model, and LLM pipeline are all provided.
**Your job is to implement each stage of the pipeline.**

In [19]:
# ========== PROVIDED: Document Corpus ==========
# A set of short ML-focused documents. Your chatbot will answer questions about these.
DOCUMENTS = [
    """The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'
    by Vaswani et al. at Google Brain. It replaced recurrent neural networks with self-attention
    mechanisms, allowing the model to process all tokens in parallel rather than sequentially.
    This enabled training on much larger datasets and dramatically improved performance on
    sequence-to-sequence tasks like translation and summarisation.""",

    """BERT (Bidirectional Encoder Representations from Transformers) is an encoder-only
    Transformer introduced by Google in 2018. It is trained using Masked Language Modelling,
    where 15% of input tokens are randomly masked and the model must predict them using
    both left and right context simultaneously. BERT is primarily used for text classification,
    named entity recognition, and extractive question answering.""",

    """GPT (Generative Pre-trained Transformer) is a decoder-only Transformer developed by
    OpenAI. It is trained on next-token prediction: given a sequence of tokens, predict the
    next one. GPT models use causal attention, meaning each token can only attend to previous
    tokens. This makes them naturally suited for text generation tasks. GPT-4, released in
    2023, is estimated to have around 1 trillion parameters.""",

    """RLHF (Reinforcement Learning from Human Feedback) is the training technique used to
    align language models with human preferences. It has three stages: supervised fine-tuning
    on instruction-following examples, training a reward model on human comparisons of
    model outputs, and using PPO (Proximal Policy Optimisation) to fine-tune the language
    model to maximise the reward model's scores. ChatGPT and Claude were both trained with RLHF.""",

    """RAG (Retrieval-Augmented Generation) is a technique that combines a language model with
    an external knowledge retrieval system. Documents are chunked, embedded using a sentence
    encoder, and stored in a vector database. At inference time, the user's query is embedded,
    the most similar document chunks are retrieved, and these chunks are injected into the
    LLM's prompt as context. This allows the model to answer questions about documents it was
    never trained on, without the hallucination risk of relying solely on parametric memory.""",

    """Word2Vec is a family of models introduced by Mikolov et al. at Google in 2013 for
    learning dense word embeddings. The two architectures are Skip-gram (predicts context
    words given a centre word) and CBOW (predicts a centre word given its context words).
    Words with similar meanings cluster together in the embedding space, enabling vector
    arithmetic such as: king - man + woman ≈ queen. Word2Vec embeddings are static —
    each word has one fixed vector regardless of context.""",

    """Few-shot prompting is a technique where 2-5 examples of a task are included directly
    in the prompt, allowing the LLM to infer the task format without any weight updates.
    Chain-of-Thought (CoT) prompting extends this by including step-by-step reasoning in
    the examples, which dramatically improves performance on arithmetic, logical, and
    multi-step reasoning tasks. Simply appending 'Let's think step by step' to a prompt
    enables zero-shot CoT reasoning.""",

    """Vector databases store high-dimensional embeddings and support efficient approximate
    nearest-neighbour search. Given a query vector, they return the K stored vectors with
    the highest cosine similarity or dot product. Popular options include FAISS (Facebook
    AI Similarity Search, open-source), Pinecone (managed cloud service), Chroma (lightweight,
    local), and Weaviate (open-source with a GraphQL API). FAISS uses techniques like
    product quantisation and inverted file indices to search billions of vectors in milliseconds.""",
]

print(f'Corpus: {len(DOCUMENTS)} documents loaded.')

Corpus: 8 documents loaded.


In [20]:
# ========== PROVIDED: Models ==========
# Do NOT change these — they are fixed for the assignment.

print('Loading embedding model...')
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = pipeline(
    'feature-extraction',
    model=EMBED_MODEL,
    tokenizer=EMBED_MODEL,
    device=-1,   # -1 = CPU; change to 0 for GPU if available
)
EMBED_DIM = 384
print(f'Embedding model loaded. Output dim: {EMBED_DIM}')

print('Loading text generation model...')
generator = pipeline(
    'text2text-generation',
    model='google/flan-t5-base',
    device=-1,
)
print('Generation model loaded.')

Loading embedding model...


Device set to use cpu


Embedding model loaded. Output dim: 384
Loading text generation model...


Device set to use cpu


Generation model loaded.


---
### B1 — Embed a Single Text  `[TODO]`

The embedding pipeline returns a nested list. Your job is to:
1. Run the text through `embedder`
2. Mean-pool over the token dimension (average across all token vectors)
3. Return a 1D numpy array of shape `(EMBED_DIM,)` with dtype `float32`

In [21]:
def embed_text(text: str) -> np.ndarray:
    """
    Embed a single string into a dense vector.

    Args:
        text: input string
    Returns:
        numpy array of shape (EMBED_DIM,), dtype float32

    TODO:
      1. raw = embedder(text)             # returns a nested list: [[[f, f, f, ...], ...], ...]
      2. Convert raw[0] to a 2D numpy array of shape (num_tokens, EMBED_DIM)
      3. Mean-pool along axis 0 -> shape (EMBED_DIM,)
      4. Return as float32
    """
    # YOUR CODE HERE
    raw = embedder(text)
    # Convert raw[0] to a 2D numpy array
    token_embeddings = np.array(raw[0])
    # Mean-pool along axis 0
    mean_pooled_embedding = np.mean(token_embeddings, axis=0)
    # Return as float32
    return mean_pooled_embedding.astype(np.float32)


# ===== Sanity Check =====
vec = embed_text('Hello world')
assert vec is not None, 'embed_text() returned None'
assert isinstance(vec, np.ndarray), f'Expected np.ndarray, got {type(vec)}'
assert vec.shape == (EMBED_DIM,), f'Expected shape ({EMBED_DIM},), got {vec.shape}'
assert vec.dtype == np.float32, f'Expected float32, got {vec.dtype}'
print(f'PASS: embed_text() works! Shape: {vec.shape}')

PASS: embed_text() works! Shape: (384,)


---
### B2 — Chunk Documents  `[TODO]`

Split a list of long documents into smaller, overlapping chunks.
This ensures no single chunk is too long for the model and that
information at chunk boundaries is not lost.

In [22]:
def chunk_documents(
    documents: list[str],
    chunk_size: int = 200,
    overlap: int = 40,
) -> list[str]:
    """
    Split each document into overlapping character-level chunks.

    Args:
        documents: list of document strings
        chunk_size: target size of each chunk in characters
        overlap: number of characters to overlap between consecutive chunks

    Returns:
        flat list of chunk strings

    TODO:
      For each document:
        Use a sliding window of size chunk_size with step (chunk_size - overlap).
        i.e., start positions: 0, chunk_size-overlap, 2*(chunk_size-overlap), ...
        Each chunk is doc[i : i + chunk_size].
        Strip whitespace from each chunk and skip empty ones.
      Return the flat list of all chunks across all documents.
    """
    # YOUR CODE HERE
    all_chunks = []
    for doc in documents:
        start = 0
        while start < len(doc):
            end = start + chunk_size
            chunk = doc[start:end].strip()
            if chunk:
                all_chunks.append(chunk)
            start += (chunk_size - overlap)
            if chunk_size - overlap <= 0 and start < len(doc): # Handle cases where overlap >= chunk_size to avoid infinite loops
                start = len(doc) # Break loop if we can't advance

    return all_chunks


# ===== Sanity Check =====
test_chunks = chunk_documents(['a' * 500], chunk_size=200, overlap=40)
assert test_chunks is not None, 'chunk_documents() returned None'
assert len(test_chunks) > 1, 'Should produce multiple chunks for a 500-char doc'
assert all(len(c) <= 200 for c in test_chunks), 'A chunk exceeded chunk_size'
print(f'PASS: chunk_documents() works! {len(test_chunks)} chunks from 500-char doc')

chunks = chunk_documents(DOCUMENTS)
print(f'Full corpus: {len(DOCUMENTS)} documents -> {len(chunks)} chunks')
print(f'Sample chunk: "{chunks[0][:120]}..."')

PASS: chunk_documents() works! 4 chunks from 500-char doc
Full corpus: 8 documents -> 27 chunks
Sample chunk: "The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'
    by Vaswani et al. at Googl..."


---
### B3 — Build the FAISS Index  `[TODO]`

Embed all chunks and store them in a FAISS index for fast nearest-neighbour retrieval.

In [23]:
def build_index(chunks: list[str]) -> tuple[faiss.Index, np.ndarray]:
    """
    Embed all chunks and build a FAISS inner-product index.

    Args:
        chunks: list of text chunks
    Returns:
        (index, embeddings) where:
          - index is a faiss.IndexFlatIP ready for search
          - embeddings is a float32 array of shape (len(chunks), EMBED_DIM)

    TODO:
      1. Embed each chunk using embed_text() -> stack into shape (N, EMBED_DIM)
         Hint: np.array([embed_text(c) for c in chunks])
      2. L2-normalise the embeddings in-place:
         faiss.normalize_L2(embeddings)   # modifies in-place
         (after normalisation, inner product = cosine similarity)
      3. Create index = faiss.IndexFlatIP(EMBED_DIM)
      4. Add embeddings to the index: index.add(embeddings)
      5. Return (index, embeddings)
    """
    # YOUR CODE HERE
    embeddings = np.array([embed_text(c) for c in chunks]).astype(np.float32)
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(embeddings)
    return (index, embeddings)


print('Building index (this takes ~30 seconds on CPU)...')
index, embeddings = build_index(chunks)

# ===== Sanity Check =====
assert index is not None, 'build_index() returned None'
assert index.ntotal == len(chunks), f'Expected {len(chunks)} vectors in index, got {index.ntotal}'
assert embeddings.shape == (len(chunks), EMBED_DIM)
print(f'PASS: Index built! {index.ntotal} vectors of dim {EMBED_DIM}')

Building index (this takes ~30 seconds on CPU)...
PASS: Index built! 27 vectors of dim 384


---
### B4 — Retrieve Relevant Chunks  `[TODO]`

Given a question, embed it and find the most semantically similar chunks.

In [24]:
import faiss

def retrieve(
    question: str,
    index: faiss.Index,
    chunks: list[str],
    k: int = 3,
) -> list[tuple[str, float]]:
    """
    Retrieve the top-k most relevant chunks for a question.

    Args:
        question: user's query string
        index: FAISS index containing chunk embeddings
        chunks: original list of chunk strings (same order as index)
        k: number of chunks to retrieve
    Returns:
        list of (chunk_text, similarity_score) tuples, sorted by score descending

    TODO:
      1. Embed the question using embed_text() -> shape (EMBED_DIM,)
      2. Reshape to (1, EMBED_DIM) and convert to float32
      3. L2-normalise: faiss.normalize_L2(query_vec)
      4. Search: scores, indices = index.search(query_vec, k)
         scores and indices are shape (1, k) — take [0] to get 1D arrays
      5. Return [(chunks[idx], score) for idx, score in zip(indices[0], scores[0])]
    """
    # YOUR CODE HERE
    # 1. Embed the question
    query_vec = embed_text(question)
    # 2. Reshape to (1, EMBED_DIM) and convert to float32
    query_vec = query_vec.reshape(1, -1).astype(np.float32)
    # 3. L2-normalise
    faiss.normalize_L2(query_vec)
    # 4. Search
    scores, indices = index.search(query_vec, k)
    # 5. Return [(chunks[idx], score) for idx, score in zip(indices[0], scores[0])]
    return [(chunks[idx], float(score)) for idx, score in zip(indices[0], scores[0])]


# ===== Sanity Check =====
test_results = retrieve('What is BERT?', index, chunks, k=3)
assert test_results is not None, 'retrieve() returned None'
assert len(test_results) == 3, f'Expected 3 results, got {len(test_results)}'
assert isinstance(test_results[0][0], str), 'First element of each tuple should be a string'
assert isinstance(test_results[0][1], float), 'Second element of each tuple should be a float'
print('PASS: retrieve() works!')
print('\nTop 3 chunks for "What is BERT?":')
for i, (chunk, score) in enumerate(test_results):
    print(f'\n[{i+1}] Score: {score:.4f}')
    print(textwrap.fill(chunk[:200], width=80))

PASS: retrieve() works!

Top 3 chunks for "What is BERT?":

[1] Score: 0.6343
BERT (Bidirectional Encoder Representations from Transformers) is an encoder-
only     Transformer introduced by Google in 2018. It is trained using Masked
Language Modelling,     where 15% of input to

[2] Score: 0.5801
age Modelling,     where 15% of input tokens are randomly masked and the model
must predict them using     both left and right context simultaneously. BERT is
primarily used for text classification,

[3] Score: 0.2323
isation) to fine-tune the language     model to maximise the reward model's
scores. ChatGPT and Claude were both trained with RLHF.


---
### B5 — Build the RAG Prompt  `[TODO]`

Format the retrieved chunks and the user's question into a prompt the LLM can use.

In [25]:
def build_prompt(question: str, context_chunks: list[tuple[str, float]]) -> str:
    """
    Build an LLM prompt from a question and retrieved context chunks.

    Args:
        question: user's question
        context_chunks: list of (chunk_text, score) tuples from retrieve()
    Returns:
        formatted prompt string

    TODO:
      Build a prompt with this structure:

      Answer the question using only the context below.
      If the answer is not in the context, say "I don't know".

      Context:
      [Source 1]: <chunk text>

      [Source 2]: <chunk text>

      [Source 3]: <chunk text>

      Question: <question>
      Answer:

    Each source should be on its own line, separated by a blank line.
    """
    # YOUR CODE HERE
    prompt_header = "Answer the question using only the context below.\nIf the answer is not in the context, say \"I don't know\".\n\nContext:"

    sources = []
    for i, (chunk, score) in enumerate(context_chunks):
        sources.append(f"[Source {i+1}]: {chunk}")

    context_text = "\n\n".join(sources)

    final_prompt = f"{prompt_header}\n{context_text}\n\nQuestion: {question}\nAnswer:"
    return final_prompt


# ===== Sanity Check =====
test_chunks_for_prompt = [('Transformers were introduced in 2017.', 0.95),
                           ('BERT is an encoder model.', 0.80)]
prompt = build_prompt('When were Transformers introduced?', test_chunks_for_prompt)
assert prompt is not None, 'build_prompt() returned None'
assert 'Source 1' in prompt, 'Prompt should contain [Source 1]'
assert 'Source 2' in prompt, 'Prompt should contain [Source 2]'
assert 'When were Transformers introduced?' in prompt, 'Prompt must contain the question'
assert 'Answer:' in prompt, 'Prompt must end with Answer:'
print('PASS: build_prompt() works!')
print('\nSample prompt:')
print('-' * 60)
print(prompt)
print('-' * 60)

PASS: build_prompt() works!

Sample prompt:
------------------------------------------------------------
Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context:
[Source 1]: Transformers were introduced in 2017.

[Source 2]: BERT is an encoder model.

Question: When were Transformers introduced?
Answer:
------------------------------------------------------------


---
### B6 — Generate an Answer  `[TODO]`

Pass the prompt to the LLM and extract the generated answer.

In [26]:
def generate_answer(prompt: str, max_new_tokens: int = 150) -> str:
    """
    Generate an answer from the LLM given a RAG prompt.
    """
    # 1. Call generator
    result = generator(prompt, max_new_tokens=max_new_tokens)
    # 2. Extract and strip text
    answer = result[0]['generated_text']
    return answer.strip()

# ===== Sanity Check =====
test_prompt = 'Answer in one sentence. What is 2 + 2? Answer:'
answer = generate_answer(test_prompt, max_new_tokens=20)
assert answer is not None, 'generate_answer() returned None'
assert isinstance(answer, str), f'Expected str, got {type(answer)}'
assert len(answer) > 0, 'Answer is empty'
print(f'PASS: generate_answer() works! Sample output: "{answer}"')

PASS: generate_answer() works! Sample output: "2 + 2"


---
### B7 — The Full RAG Pipeline  `[TODO]`

Wire together all the pieces you've built into one function.

In [27]:
def rag_answer(question: str, index: faiss.Index, chunks: list[str], k: int = 3) -> str:
    """
    Full RAG pipeline: question -> retrieve -> prompt -> generate -> answer.
    """
    # 1. Retrieve top-k chunks
    retrieved_chunks = retrieve(question, index, chunks, k=k)
    # 2. Build prompt
    prompt = build_prompt(question, retrieved_chunks)
    # 3. Generate answer
    answer = generate_answer(prompt)
    return answer

# ===== Sanity Check =====
test_questions = [
    'What is BERT and what is it used for?',
    'How does RLHF work?',
]

print('========== RAG PIPELINE TEST ==========')
for q in test_questions:
    answer = rag_answer(q, index, chunks)
    assert answer is not None and len(answer) > 0, f'Got empty answer for: {q}'
    print(f'\nQ: {q}')
    print(f'A: {textwrap.fill(answer, width=80)}')

print('\nPASS: Full RAG pipeline works!')

========== RAG PIPELINE TEST ==========

Q: What is BERT and what is it used for?
A: BERT is primarily used for text classification, [Source 3]: RAG (Retrieval-
Augmented Generation) is a technique that combines a language model with an
external knowledge retrieval system. Documents are chunked, embedded using a
sentence encoder, and st

Q: How does RLHF work?
A: It has three stages: supervised fine-tuning on instruction-fol [Source 2]:
isation) to fine-tune the language model to maximise the reward model's scores.
ChatGPT and Claude were both trained with RLHF. [Source 3]: rieved, and these
chunks are injected into the LLM's prompt as context. This allows the model to
answer questions about documents it was never trained on, without the
hallucination risk of rel

PASS: Full RAG pipeline works!


---
### B8 — The Console Chatbot  `[TODO]`

Now build the interactive loop. This is the payoff — put it all together.

In [28]:
def run_chatbot(index: faiss.Index, chunks: list[str]) -> None:
    """
    Run an interactive console chatbot loop.
    """
    print("\n" + "─"*40)
    print("ML Chatbot (type 'quit' to exit)")
    print("─"*40)

    while True:
        question = input('You: ').strip()
        if not question:
            continue
        if question.lower() in ['quit', 'exit']:
            print('Bot: Goodbye!')
            break

        print('Bot: Thinking...')
        answer = rag_answer(question, index, chunks)
        print(f'Bot: {answer}\n')

---
## Part C — The Challenge

Choose **one** of the two extensions below. Both are open-ended — there is no single right answer. You are graded on your approach, code quality, and reflection.

---
### Option 1 — Multi-Turn Memory

Right now, each question is answered independently — the chatbot has no memory of previous turns.

**Task:** Modify `run_chatbot()` to maintain a conversation history, and include the last N turns in the prompt so the bot can answer follow-up questions coherently.

Example:
```
You: What is BERT?
Bot: BERT is an encoder-only Transformer...

You: What was it trained on?    ← requires memory of previous turn to answer
Bot: BERT was trained using Masked Language Modelling...
```

**Hint:** Modify `build_prompt()` to accept an optional `history: list[tuple[str,str]]` parameter (list of (question, answer) pairs) and prepend it to the context.

In [31]:
def build_prompt_with_history(
    question: str,
    context_chunks: list[tuple[str, float]],
    history: list[tuple[str, str]],
    max_history: int = 3,
) -> str:
    """
    Build a RAG prompt that includes recent conversation history.
    """
    prompt_header = "Answer the question using only the context below.\nIf the answer is not in the context, say \"I don't know\".\n"

    # Add History
    history_text = ""
    if history:
        history_text = "Recent Conversation History:\n"
        for h_q, h_a in history[-max_history:]:
            history_text += f"User: {h_q}\nBot: {h_a}\n"
        history_text += "\n"

    # Add Context
    sources = []
    for i, (chunk, score) in enumerate(context_chunks):
        sources.append(f"[Source {i+1}]: {chunk}")
    context_text = "Context:\n" + "\n\n".join(sources)

    final_prompt = f"{prompt_header}\n{history_text}{context_text}\n\nQuestion: {question}\nAnswer:"
    return final_prompt

def run_chatbot_with_memory(index: faiss.Index, chunks: list[str]) -> None:
    """
    Interactive loop with conversation memory.
    """
    print("\n" + "═"*40)
    print("ML Chatbot with Memory (type 'quit' to exit)")
    print("═"*40)

    history = []

    while True:
        try:
            question = input('You: ').strip()
        except EOFError:
            break

        if not question:
            continue
        if question.lower() in ['quit', 'exit']:
            print('Bot: Goodbye!')
            break

        print('Bot: Thinking...')
        # 1. Retrieve context
        retrieved_chunks = retrieve(question, index, chunks, k=3)
        # 2. Build prompt with history
        prompt = build_prompt_with_history(question, retrieved_chunks, history)
        # 3. Generate
        answer = generate_answer(prompt)

        print(f'Bot: {answer}\n')
        # 4. Update history
        history.append((question, answer))

# To start the chatbot, run this cell.
run_chatbot_with_memory(index, chunks)


════════════════════════════════════════
ML Chatbot with Memory (type 'quit' to exit)
════════════════════════════════════════
You: What was the 2017 paper that introduced Transformers?
Bot: Thinking...
Bot: Attention Is All You Need

You: What is the difference between Skip-gram and CBOW
Bot: Thinking...
Bot: Word2Vec is a family of models introduced by Mikolov et al. at Google in 2013 for learning dense word embeddings. The two architectures are Skip-gram (predicts context words given a centre word) and CBOW (predicts a centre word given its context words). Words with similar meanings cluster together in the embedding space, enabling vector arith

You: quit
Bot: Goodbye!


---
### Option 2 — Source Citation

Right now, the chatbot gives an answer but cites no sources. Users have no way to verify the answer.

**Task:** After each answer, print the top retrieved source chunks (with their similarity scores) so the user can see *where* the answer came from.

**Bonus:** Track which document each chunk came from (add a `doc_id` to each chunk during chunking) and display `[Source: Document 3]` alongside each chunk.

In [33]:
import faiss
import numpy as np

def chunk_documents_with_ids(
    documents: list[str],
    chunk_size: int = 200,
    overlap: int = 40,
) -> list[dict]:
    """
    Split documents into chunks while tracking their source document ID.
    Returns a list of dicts: {'text': str, 'doc_id': int}
    """
    chunks_with_metadata = []
    for doc_idx, doc in enumerate(documents):
        start = 0
        while start < len(doc):
            end = start + chunk_size
            chunk_text = doc[start:end].strip()
            if chunk_text:
                chunks_with_metadata.append({
                    'text': chunk_text,
                    'doc_id': doc_idx
                })
            start += (chunk_size - overlap)
            if chunk_size - overlap <= 0: break
    return chunks_with_metadata

def run_chatbot_with_citations(index: faiss.Index, chunks_metadata: list[dict]) -> None:
    """
    Chatbot loop that prints retrieved sources and scores.
    """
    print("\n" + "═"*40)
    print("ML Chatbot with Citations (type 'quit' to exit)")
    print("═"*40)

    # Extract just the text for the standard retrieve function logic
    texts = [c['text'] for c in chunks_metadata]

    while True:
        try:
            question = input('You: ').strip()
        except EOFError:
            break

        if not question or question.lower() in ['quit', 'exit']:
            print('Bot: Goodbye!')
            break

        print('Bot: Thinking...')
        # 1. Retrieve
        results = retrieve(question, index, texts, k=3)

        # 2. Generate Answer
        prompt = build_prompt(question, results)
        answer = generate_answer(prompt)

        print(f'Bot: {answer}\n')

        # 3. Print Citations
        print("Sources:")
        for i, (text, score) in enumerate(results):
            # Find the doc_id from metadata (matching by text index)
            # In a real system, retrieve() would return indices directly.
            print(f" [{i+1}] (score: {score:.4f}) {text[:100]}...")
        print()

# Prepare metadata-aware chunks and rebuild index for consistency
chunks_meta = chunk_documents_with_ids(DOCUMENTS)
index_meta, _ = build_index([c['text'] for c in chunks_meta])

# Start the chatbot
run_chatbot_with_citations(index_meta, chunks_meta)


════════════════════════════════════════
ML Chatbot with Citations (type 'quit' to exit)
════════════════════════════════════════
You: how are you
Bot: Thinking...
Bot: I don't know

Sources:
 [1] (score: 0.1302) s of context....
 [2] (score: 0.1183) vised fine-tuning
    on instruction-following examples, training a reward model on human comparison...
 [3] (score: 0.1027) isation) to fine-tune the language
    model to maximise the reward model's scores. ChatGPT and Clau...

You: quit
Bot: Goodbye!


## Reflection (Required)

1. **Which chunks did your system retrieve for the question "How was ChatGPT trained?"? Were they the right ones?**
It retrieved chunks related to RLHF (Reinforcement Learning from Human Feedback), specifically discussing supervised fine-tuning and reward models. These were the correct ones because ChatGPT's alignment is the primary focus of that specific document in our corpus.

2. **Try asking your bot a question whose answer is NOT in the corpus... What does it say?**
When asked about the weather or who I am, it correctly said "I don't know." This is the intended behavior for a grounded RAG system. To fix it so it can answer general questions, I could either add a 'General Knowledge' document to the corpus or modify the prompt to allow the LLM to use its own internal training data when context is missing.

3. **What is one concrete limitation of your current chunking strategy?**
Our character-level sliding window can cut words in half or split a single important sentence across two different chunks, causing a loss of semantic meaning. I would improve this by using a recursive character splitter or a sentence-aware splitter that respects punctuation boundaries.

4. **Which Part C option did you implement?**
I implemented Option 1: Multi-Turn Memory. I modified the chatbot loop to maintain a 'history' list and updated the prompt builder to prepend previous turns to the current query. If I had more time, I would implement a 'summarization' step for the history to prevent the prompt from becoming too long after many turns.

5. **What is the fundamental difference between TF-IDF and embedding-based retrieval?**
TF-IDF relies on exact keyword matching and term frequency, meaning it fails if the user uses synonyms that aren't in the text. In contrast, embedding-based retrieval uses semantic meaning; it represents text as vectors in a high-dimensional space where concepts like 'king' and 'royal' are close together even if they share no common letters.